# 01 — Data Cleaning & Preprocessing Pipeline
## Corporación Favorita Grocery Sales Forecasting

### Purpose of this Notebook
This notebook establishes the foundational clean data layer for the entire forecasting project:
1. **Universal Path Resolution:** Works seamlessly in both local environments and Google Colab.
2. **Missing Value Imputation:** Resolves weekend oil trading gaps via temporal resampling and interpolation.
3. **Calendar Sanitization:** Cleans the multi-tier holiday dataset (National, Regional, Local) using locale priority ranking to prevent row explosion.
4. **Relational Master Merge:** Combines historical store sales with store metadata, oil prices, and holiday events.
5. **Categorical Encoding:** Fits and serializes `LabelEncoder` objects for store types, product families, clusters, and holiday types.
6. **Dataset Serialization:** Exports `clean_merged.csv` into `data/`, which directly feeds downstream EDA, classical time-series, and feature engineering notebooks.


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

# --- 1. Universal Path Resolution (Colab & Local) ---
if os.path.exists('/content/drive/MyDrive/Sales-Forecasting_Project'):
    BASE_DIR = Path('/content/drive/MyDrive/Sales-Forecasting_Project/sales-forecasting')
else:
    cwd = Path.cwd().resolve()
    BASE_DIR = cwd.parent if cwd.name == 'notebooks' else cwd

DATA_DIR = BASE_DIR / 'data'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'
MODELS_DIR = BASE_DIR / 'models'
REPORTS_DIR = BASE_DIR / 'reports'

for directory in [DATA_DIR, PROCESSED_DATA_DIR, MODELS_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Base Directory:    {BASE_DIR}")
print(f"Data Directory:    {DATA_DIR}")
print(f"Models Directory:  {MODELS_DIR}")
print(f"Reports Directory: {REPORTS_DIR}")


### 2. Ingesting Raw Data
We load the raw CSV files provided in the dataset:
- `train.csv`: Historical daily sales records across 54 stores and 33 product families.
- `stores.csv`: Metadata for each store (city, state, store type, cluster).
- `oil.csv`: Daily West Texas Intermediate (WTI) crude oil prices.
- `holidays_events.csv`: National, regional, and local Ecuadorian holidays and transfer events.
- `transactions.csv`: Daily customer transaction counts per store.


In [ ]:
print("Loading raw datasets...")
df_train = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['date'])
df_oil = pd.read_csv(DATA_DIR / 'oil.csv', parse_dates=['date'])
df_stores = pd.read_csv(DATA_DIR / 'stores.csv')
df_holidays = pd.read_csv(DATA_DIR / 'holidays_events.csv', parse_dates=['date'])
df_transactions = pd.read_csv(DATA_DIR / 'transactions.csv', parse_dates=['date'])

print(f"df_train:        {df_train.shape[0]:,} rows x {df_train.shape[1]} cols | Dates: {df_train['date'].min().date()} to {df_train['date'].max().date()}")
print(f"df_stores:       {df_stores.shape[0]:,} stores")
print(f"df_oil:          {df_oil.shape[0]:,} rows x {df_oil.shape[1]} cols")
print(f"df_holidays:     {df_holidays.shape[0]:,} events")
print(f"df_transactions: {df_transactions.shape[0]:,} transaction logs")


### 3. Date Coercion and Integrity Validation
Ensure all temporal columns are strictly typed as datetime, removing any invalid or unparseable records.


In [ ]:
df_train['date'] = pd.to_datetime(df_train['date'], errors='coerce')
df_oil['date'] = pd.to_datetime(df_oil['date'], errors='coerce')
df_holidays['date'] = pd.to_datetime(df_holidays['date'], errors='coerce')
df_transactions['date'] = pd.to_datetime(df_transactions['date'], errors='coerce')

# Drop invalid dates if any
df_train = df_train.dropna(subset=['date'])
df_oil = df_oil.dropna(subset=['date'])
df_holidays = df_holidays.dropna(subset=['date'])

print("Date validation complete. All temporal indices confirmed valid.")


### 4. Oil Price Imputation & Daily Resampling
Crude oil prices are recorded only on business trading days, leading to gaps over weekends and holidays.
We enforce a continuous daily frequency (`resample('D')`), apply linear interpolation across trading breaks, and backfill (`bfill`) the first missing row.


In [ ]:
print(f"Missing oil prices before resampling & interpolation: {df_oil['dcoilwtico'].isnull().sum()}")

# Resample to full continuous calendar days
df_oil = df_oil.set_index('date').resample('D').mean().interpolate(method='linear').reset_index()

# Backfill any leading missing days (e.g. initial trading holidays)
df_oil['dcoilwtico'] = df_oil['dcoilwtico'].bfill()

print(f"Missing oil prices after interpolation: {df_oil['dcoilwtico'].isnull().sum()}")
df_oil.head()


### 5. Holiday Calendar Deduplication & Locale Hierarchy
In Ecuador, some days feature multiple concurrent events (e.g. a local city celebration coinciding with a national independence day).
A naive relational merge will cause a **Cartesian row explosion**, duplicating sales rows.
We resolve this by creating a locale hierarchy (`National: 1, Regional: 2, Local: 3`) and keeping only the primary event per calendar day.


In [ ]:
print(f"Total holiday rows before deduplication: {len(df_holidays)}")

# Assign priority ranking
locale_priority = {'National': 1, 'Regional': 2, 'Local': 3}
df_holidays['locale_rank'] = df_holidays['locale'].map(locale_priority).fillna(99)

# Sort by date and priority, then keep the highest-priority event per date
holidays_clean = df_holidays.sort_values(['date', 'locale_rank']).drop_duplicates(subset=['date'], keep='first').copy()

print(f"Unique calendar days with holidays: {len(holidays_clean)}")
holidays_clean.head()


### 6. Relational Master Merge
Now we perform left joins on `df_train` to preserve all sales records while attaching store attributes, continuous oil prices, and cleaned holiday details.


In [ ]:
print("Executing Master Relational Merge...")
merged = df_train.merge(df_stores, on='store_nbr', how='left')
merged = merged.merge(df_oil, on='date', how='left')
merged = merged.merge(holidays_clean[['date', 'type', 'locale', 'transferred']], on='date', how='left')

# Rename merged type columns for clarity
merged = merged.rename(columns={'type_x': 'store_type', 'type_y': 'holiday_type'})
merged['is_holiday'] = merged['holiday_type'].notnull().astype(int)
merged['holiday_type'] = merged['holiday_type'].fillna('None')

print(f"Master merge complete! Output shape: {merged.shape[0]:,} rows x {merged.shape[1]} columns")
merged.head()


### 7. Categorical Encoding & Model Artifact Serialization
Categorical variables (`family`, `store_type`, `cluster`, `holiday_type`) are encoded into integers using `LabelEncoder`.
Crucially, these encoders are pickled and saved to `models/` so that runtime inference and dashboard lookups can consistently decode labels.


In [ ]:
from sklearn.preprocessing import LabelEncoder

categorical_columns = ['family', 'store_type', 'cluster', 'holiday_type']
encoders = {}

for col in categorical_columns:
    le = LabelEncoder()
    merged[col] = le.fit_transform(merged[col].astype(str))
    encoders[col] = le
    
    # Save encoder
    encoder_path = MODELS_DIR / f"{col}_encoder.pkl"
    with open(encoder_path, 'wb') as f:
        pickle.dump(le, f)
    print(f"Saved {col}_encoder.pkl ({len(le.classes_)} classes) -> {encoder_path}")


### 8. Data Sanity Checks & Zero-Sales Profiling
Check zero-sales frequency and confirm no negative sales values exist.


In [ ]:
zero_sales = (merged['sales'] == 0).sum()
pct_zero = (zero_sales / len(merged)) * 100
negative_sales = (merged['sales'] < 0).sum()

print(f"Zero sales count:     {zero_sales:,} ({pct_zero:.2f}%)")
print(f"Negative sales count: {negative_sales:,}")
assert negative_sales == 0, "Data integrity error: negative sales detected!"

# Verify date range
print(f"Cleaned date range:   {merged['date'].min().date()} to {merged['date'].max().date()}")


### 9. Saving Clean Master Dataset
We export the final clean dataset to `data/clean_merged.csv`.
This file serves as the standardized input for:
- `02_exploratory_data_analysis.ipynb`
- `03_feature_engineering.ipynb`
- `04_arima_forecasting.ipynb`
- `05_sarima_forecasting.ipynb`
- `06_prophet_forecasting.ipynb`


In [ ]:
output_file = DATA_DIR / 'clean_merged.csv'
print(f"Saving clean dataset to: {output_file}...")
merged.to_csv(output_file, index=False)
print(f"Successfully serialized clean_merged.csv ({output_file.stat().st_size / (1024 * 1024):.2f} MB)")
print("Done! You may now proceed to 02_exploratory_data_analysis.ipynb or 03_feature_engineering.ipynb.")
